# 兴业 · 单资产尾部风险预警复现

复现兴业证券金工《基于宏观相似、动量趋势和尾部风险控制的全球资产配置研究》（2025-06-26）第 4 章
「尾部风险控制」：为每类资产构建一个预警信号，检验信号后 1/2/3/5/10 日的平均收益，
并回测「信号后空仓 5 个交易日」的择时策略。原文回测区间 2010-01-04 至 2025-04-30。

## 原文的信号

| 资产 | 信号（「超过 N 的 99% 分位」指高于过去 N 的 99% 分位数；「上升」指近 5 日斜率为正） |
|---|---|
| 沪深300 / 中证500 | 股债性价比（全A股息率 − 10年国债收益率）低于过去 4 年 10% 分位 **且** 指数近 5 日下跌 **且** PcaScore 近 21 日均值 > 近 63 日均值 |
| 恒生指数 | 美元兑离岸人民币超过 2 年 99% 分位 **且** 美国 10 年实际利率超过 1 年 99% 分位并上升 |
| 标普500 | VIX 超过 1 年 99% 分位并上升 **或** 实际利率超过 2 年 99% 分位并上升 |
| 纳斯达克100 | 同标普500，VIX 换成 VXN |
| 黄金 | 实际利率超过 3 个月 99% 分位并上升 |
| 原油 | OVX 超过 1 年 99% 分位并上升 |
| 南华商品指数 | 美元指数超过 2 年 99% 分位并上升（**未复现**：免费的商品指数数据只有近 4 年） |

## 复现口径

| 项 | 原文（Wind） | 本复现 |
|---|---|---|
| 全A股息率（原文为「改进后」） | 万得全A | 乐咕乐股「上证A股」股息率 |
| PcaScore | 指数成分股收益的前 2 / 前 20 大主成分方差占比 | 申万一级行业代替成分股（固定用 2010 年已存在的 16 个行业），21 日窗口* |
| 美元兑离岸人民币 | USD/CNH | 美元兑在岸人民币（FRED） |
| 恒生指数 | 恒生指数 | 盈富基金（02800.HK） |
| 黄金 | 未说明 | GLD（美元）；另试国内金价 |
| 原油 | ICE 布伦特原油指数 | 布伦特现货（FRED） |
| 其余 | — | 美国实际利率、VIX、VXN、OVX 均取自 FRED；标普500 取自新浪；纳指100 取自 FRED |

\* 原文只取前 20 个主成分，最自然的解释是窗口约 20 个交易日（此时协方差矩阵至多 20 个非零特征值），据此取 21 日；行业数量随时间增加会让 PcaScore 水平跳变，所以固定行业集合。

## 0 · 环境与数据

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%run ../../01_methods.ipynb

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
plt.rcParams.update({
    "figure.figsize": (13, 4),
    "axes.unicode_minus": False,
    "font.sans-serif": ["PingFang SC", "Heiti TC", "Arial Unicode MS", "SimHei", "DejaVu Sans"],
})

L = load_series
real_yield, vix, vxn, ovx, usdcny = L("美国10年实际利率"), L("VIX"), L("VXN"), L("OVX"), L("美元兑人民币")
div_yield, cn10y = L("上证A股股息率"), L("中国10年国债收益率")
PRICE = {"沪深300": L("沪深300"), "中证500": L("中证500"), "恒生": L("恒生指数(盈富基金)"),
         "标普500": L("标普500"), "纳指100": L("纳斯达克100"), "黄金": L("GLD"), "原油": L("布伦特原油")}

sw = load_sw_industries()
SW16 = [c for c in sw if sw[c].first_valid_index() <= pd.Timestamp("2010-01-04")]
PCA = pca_score(sw[SW16].pct_change(fill_method=None).dropna(), window=21)
print(f"PcaScore 使用 {len(SW16)} 个申万一级行业，{PCA.dropna().index.min():%Y-%m-%d} 起")

IN_SAMPLE = slice("2010-01-04", "2025-04-30")
OUT_SAMPLE = slice("2025-05-01", None)

PcaScore 使用 16 个申万一级行业，2000-02-15 起


In [2]:
def build_signals(q=0.99, scale=1.0, a_q=0.10, a_scale=1.0, pca=PCA):
    w = lambda n: int(round(n * scale))
    return {
        "沪深300": a_share_signal(div_yield, cn10y, PRICE["沪深300"], pca, value_window=int(4 * 252 * a_scale), q=a_q),
        "中证500": a_share_signal(div_yield, cn10y, PRICE["中证500"], pca, value_window=int(4 * 252 * a_scale), q=a_q),
        "恒生": hk_signal(usdcny, real_yield, fx_window=w(504), ry_window=w(252), q=q),
        "标普500": us_equity_signal(vix, real_yield, vol_window=w(252), ry_window=w(504), q=q),
        "纳指100": us_equity_signal(vxn, real_yield, vol_window=w(252), ry_window=w(504), q=q),
        "黄金": gold_signal(real_yield, window=w(63), q=q),
        "原油": oil_signal(ovx, window=w(252), q=q),
    }

SIG = build_signals()
ASSETS = list(SIG)

---
# 1 · 复现：信号后的平均收益

原文口径：逐日统计，信号日之后 h 日的平均收益（信号日重叠的都算）。

In [3]:
CIB = {"沪深300": [-0.28, -0.52, -0.94, -1.20, -1.53], "中证500": [-0.47, -0.93, -1.51, -2.10, -2.29],
       "恒生": [-0.40, -0.78, -1.49, -2.34, -3.27], "标普500": [-0.23, -0.38, -0.36, -0.46, -0.71],
       "纳指100": [-0.07, -0.14, -0.07, -0.08, -0.24], "黄金": [-0.14, -0.21, -0.18, -0.13, -0.10],
       "原油": [-0.44, -1.18, -2.36, -3.39, -3.35]}
rows = {}
for a in ASSETS:
    ev = event_study(SIG[a], PRICE[a], period=IN_SAMPLE)
    sd = align(SIG[a], PRICE[a].index).loc[IN_SAMPLE]
    rows[(a, "原文")] = dict(zip([f"{h}日" for h in HORIZONS], np.array(CIB[a]) / 100))
    rows[(a, "复现")] = {f"{h}日": v for h, v in ev.loc["预警时期"].items()} | {"信号天数": int(sd.sum()), "独立事件": count_episodes(sd)}
    rows[(a, "复现·正常时期")] = {f"{h}日": v for h, v in ev.loc["正常时期"].items()}
tab1 = pd.DataFrame(rows).T
tab1.style.format({f"{h}日": "{:+.2%}" for h in HORIZONS} | {"信号天数": "{:.0f}", "独立事件": "{:.0f}"}, na_rep="")

「独立事件」：相邻信号日间隔不超过 10 个交易日的归为同一次事件。

## 1.1 择时回测：信号后空仓 5 个交易日

In [4]:
CIB_T = {"沪深300": (2.36, -41.44, 0.42, -46.70), "中证500": (4.90, -53.47, 1.46, -65.20),
         "恒生": (2.12, -46.94, 0.09, -55.70), "标普500": (13.84, -18.11, 10.95, -33.92),
         "纳指100": (17.54, -27.10, 16.49, -35.56), "黄金": (8.77, -34.54, 7.88, -44.88),
         "原油": (5.36, -70.97, -1.79, -81.78)}
rows = {}
for a in ASSETS:
    r = PRICE[a].pct_change().dropna().loc[IN_SAMPLE]
    tr, flat = timing_returns(SIG[a], r)
    s, b = perf_stats(tr, rf=0), perf_stats(r, rf=0)
    o = CIB_T[a]
    rows[a] = {"择时年化（原文）": o[0] / 100, "择时年化（复现）": s["年化收益"],
               "基准年化（原文）": o[2] / 100, "基准年化（复现）": b["年化收益"],
               "择时回撤（原文）": o[1] / 100, "择时回撤（复现）": s["最大回撤"],
               "基准回撤（原文）": o[3] / 100, "基准回撤（复现）": b["最大回撤"],
               "空仓天数占比": flat.mean()}
pd.DataFrame(rows).T.style.format("{:.2%}")

,择时年化（原文）,择时年化（复现）,基准年化（原文）,基准年化（复现）,择时回撤（原文）,择时回撤（复现）,基准回撤（原文）,基准回撤（复现）,空仓天数占比
沪深300,2.36%,0.92%,0.42%,0.35%,-41.44%,-45.60%,-46.70%,-46.70%,2.34%
中证500,4.90%,4.30%,1.46%,1.50%,-53.47%,-57.13%,-65.20%,-65.20%,1.93%
恒生,2.12%,2.19%,0.09%,0.08%,-46.94%,-46.74%,-55.70%,-55.56%,2.91%
标普500,13.84%,12.19%,10.95%,10.71%,-18.11%,-20.54%,-33.92%,-33.92%,11.82%
纳指100,17.54%,17.01%,16.49%,16.06%,-27.10%,-27.80%,-35.56%,-35.56%,11.96%
黄金,8.77%,6.19%,7.88%,6.81%,-34.54%,-40.41%,-44.88%,-45.56%,16.17%
原油,5.36%,3.95%,-1.79%,-1.29%,-70.97%,-79.17%,-81.78%,-92.88%,5.44%


**判读**：

- 标普500、纳指100、原油与原文非常接近；中证500 比原文更强；沪深300、恒生方向一致，幅度约为原文的六成。
- 择时回测的年化收益和回撤与原文同一量级（沪深300 的择时收益明显低于原文：0.9% 对 2.4%）。
- **黄金没有复现**：信号后的收益为正，与原文相反（见第 3 节）。

---
# 2 · 统计上靠得住吗？

99% 分位的触发很稀少，信号日又扎堆出现，「平均收益」可能只是少数几次大跌撑起来的。三个检验：

1. **安慰剂**：把信号序列整体循环平移（保留信号天数和扎堆结构，只打乱时点），重复 1000 次，
   看真实信号的 5 日平均收益比多少比例的随机版本更低。
2. **去掉一次事件**：去掉对平均收益贡献最负的那一次独立事件，看结论是否还在。
3. **独立事件数**：真正互相独立的观测有多少。

In [5]:
rows = {}
for a in ASSETS:
    actual, dist = placebo_event_study(SIG[a], PRICE[a], horizon=5, n=1000, period=IN_SAMPLE)
    allm, kept, (s, e) = drop_worst_episode(SIG[a], PRICE[a], horizon=5, period=IN_SAMPLE)
    sd = align(SIG[a], PRICE[a].index).loc[IN_SAMPLE]
    rows[a] = {"独立事件": count_episodes(sd), "5日均值": actual, "随机版本中位数": np.median(dist),
               "随机版本更低的比例（≈p值）": (dist <= actual).mean(),
               "去掉最负一次事件后的5日均值": kept, "被去掉的事件": f"{s:%Y-%m-%d} ~ {e:%Y-%m-%d}"}
rob = pd.DataFrame(rows).T
rob.style.format({"独立事件": "{:.0f}", "5日均值": "{:+.2%}", "随机版本中位数": "{:+.2%}",
                  "随机版本更低的比例（≈p值）": "{:.1%}", "去掉最负一次事件后的5日均值": "{:+.2%}"})

,独立事件,5日均值,随机版本中位数,随机版本更低的比例（≈p值）,去掉最负一次事件后的5日均值,被去掉的事件
沪深300,5,-0.79%,-0.02%,13.7%,+1.09%,2015-05-28 ~ 2015-06-30
中证500,4,-2.68%,+0.17%,0.5%,+0.73%,2015-06-16 ~ 2015-06-30
恒生,7,-1.34%,+0.04%,1.0%,-0.39%,2022-09-01 ~ 2022-11-03
标普500,29,-0.38%,+0.24%,0.8%,+0.00%,2020-02-24 ~ 2020-03-18
纳指100,27,-0.10%,+0.34%,3.0%,+0.29%,2022-04-05 ~ 2022-07-08
黄金,43,+0.10%,+0.16%,39.1%,+0.25%,2013-05-08 ~ 2013-07-05
原油,15,-3.23%,+0.13%,0.0%,+0.16%,2020-02-27 ~ 2020-03-30


In [6]:
for a, r in rob.iterrows():
    tag = "显著" if r["随机版本更低的比例（≈p值）"] < 0.05 else "不显著"
    print(f"{a}：{r['独立事件']:.0f} 次独立事件，安慰剂 p≈{r['随机版本更低的比例（≈p值）']:.1%}（{tag}）；"
          f"去掉 {r['被去掉的事件']} 后 5 日均值 {r['5日均值']:+.2%} → {r['去掉最负一次事件后的5日均值']:+.2%}")

沪深300：5 次独立事件，安慰剂 p≈13.7%（不显著）；去掉 2015-05-28 ~ 2015-06-30 后 5 日均值 -0.79% → +1.09%
中证500：4 次独立事件，安慰剂 p≈0.5%（显著）；去掉 2015-06-16 ~ 2015-06-30 后 5 日均值 -2.68% → +0.73%
恒生：7 次独立事件，安慰剂 p≈1.0%（显著）；去掉 2022-09-01 ~ 2022-11-03 后 5 日均值 -1.34% → -0.39%
标普500：29 次独立事件，安慰剂 p≈0.8%（显著）；去掉 2020-02-24 ~ 2020-03-18 后 5 日均值 -0.38% → +0.00%
纳指100：27 次独立事件，安慰剂 p≈3.0%（显著）；去掉 2022-04-05 ~ 2022-07-08 后 5 日均值 -0.10% → +0.29%
黄金：43 次独立事件，安慰剂 p≈39.1%（不显著）；去掉 2013-05-08 ~ 2013-07-05 后 5 日均值 +0.10% → +0.25%
原油：15 次独立事件，安慰剂 p≈0.0%（显著）；去掉 2020-02-27 ~ 2020-03-30 后 5 日均值 -3.23% → +0.16%


**判读**：

- 安慰剂检验中，除沪深300、黄金外都显著 —— 信号确实「踩中」了某些大跌，这不是随机时点能做到的。
- 但**除恒生外，每个资产的效果都只靠一次危机**：去掉 2015 年股灾，沪深300、中证500 的 5 日均值转为正；
  去掉 2020 年 3 月，标普500 归零、原油转正；去掉 2022 年加息，纳指100 转正。
- 沪深300、中证500 15 年里只有 4–5 次独立事件，原文以「逐日统计」得到的平均收益，实质上是这几次事件的平均。
- 换句话说：这些信号能识别少数几次系统性危机，但在危机之外没有可检验的预测能力。

---
# 3 · 前视检验：美国数据与亚洲资产的时间差

美国收盘晚于亚洲收盘约 12 小时。用美国 d 日的数据（实际利率、汇率、VIX 等）给亚洲资产在 d 日收盘出信号，
等于用了收盘后才知道的信息。原文按日期对齐；这里对比「同日对齐」与「只用前一个美国交易日的数据」。
原油价格为伦敦收盘的布伦特现货，OVX 为美国收盘，同样存在几个小时的时差。

In [7]:
gold_cn = L("黄金ETF")
checks = {"恒生": (SIG["恒生"], PRICE["恒生"]), "原油": (SIG["原油"], PRICE["原油"]),
          "黄金（国内：黄金ETF 518880）": (SIG["黄金"], gold_cn), "黄金（美国：GLD）": (SIG["黄金"], PRICE["黄金"])}
rows = {}
for name, (s, p) in checks.items():
    fwd = forward_returns(p).loc[IN_SAMPLE]
    for lag in (0, 1):
        sd = align(s, fwd.index, lag=lag)
        rows[(name, "同日对齐" if lag == 0 else "滞后一天")] = {f"{h}日": fwd[sd][h].mean() for h in HORIZONS} | {"信号天数": int(sd.sum())}
pd.DataFrame(rows).T.style.format({f"{h}日": "{:+.2%}" for h in HORIZONS} | {"信号天数": "{:.0f}"})

**判读**：

- 国内金价只有在同日对齐时，信号后次日收益才为负；改为滞后一天后转为正 —— 原文黄金信号的短期负收益很可能来自这一前视。
  无论哪种金价、哪种对齐，5 日、10 日收益都不像原文那样为负。
- 恒生、原油改为滞后一天后结果变化不大（恒生 5 日 −1.34% → −1.16%，原油 −3.23% → −3.56%），前视不是它们效果的来源。

---
# 4 · 参数是不是事后挑的？

原文各资产的分位数、窗口长度、「且 / 或」组合各不相同。这里把所有分位数阈值同时换成 95% / 97.5% / 99%，
窗口长度同时乘以 0.5 / 1 / 2（A股：性价比分位 5% / 10% / 20%，回看 2 / 4 / 6 年），
看「信号后 5 日均值 − 正常时期 5 日均值」在各组参数下是否仍为负。

In [8]:
grid = {}
for q in (0.95, 0.975, 0.99):
    for scale in (0.5, 1.0, 2.0):
        sigs = build_signals(q=q, scale=scale)
        for a in ["恒生", "标普500", "纳指100", "黄金", "原油"]:
            ev = event_study(sigs[a], PRICE[a], horizons=(5,), period=IN_SAMPLE)
            grid[(a, f"分位{q:.1%}")] = grid.get((a, f"分位{q:.1%}"), {}) | {f"窗口×{scale}": ev.loc["预警时期", 5] - ev.loc["正常时期", 5]}
for a_q in (0.05, 0.10, 0.20):
    for a_scale in (0.5, 1.0, 1.5):
        sigs = build_signals(a_q=a_q, a_scale=a_scale)
        for a in ["沪深300", "中证500"]:
            ev = event_study(sigs[a], PRICE[a], horizons=(5,), period=IN_SAMPLE)
            grid[(a, f"性价比分位{a_q:.0%}")] = grid.get((a, f"性价比分位{a_q:.0%}"), {}) | {f"窗口×{a_scale}": ev.loc["预警时期", 5] - ev.loc["正常时期", 5]}
sens = pd.DataFrame(grid).T
sens.style.format("{:+.2%}", na_rep="—").background_gradient(cmap="RdBu", vmin=-0.03, vmax=0.03)

,,窗口×0.5,窗口×1.0,窗口×2.0,窗口×1.5
恒生,分位95.0%,-1.18%,-0.78%,-0.40%,—
标普500,分位95.0%,-0.35%,-0.09%,-0.12%,—
纳指100,分位95.0%,-0.28%,-0.10%,-0.14%,—
黄金,分位95.0%,-0.03%,-0.19%,-0.18%,—
原油,分位95.0%,-1.34%,-1.75%,-1.54%,—
恒生,分位97.5%,-1.23%,-1.17%,-0.49%,—
标普500,分位97.5%,-0.50%,-0.24%,+0.06%,—
纳指100,分位97.5%,-0.37%,-0.17%,+0.22%,—
黄金,分位97.5%,+0.00%,-0.20%,-0.25%,—
原油,分位97.5%,-1.86%,-2.75%,-0.50%,—


In [9]:
summary = sens.apply(lambda r: pd.Series({"参数组合数": r.notna().sum(), "为负的组合": int((r < 0).sum())}), axis=1)
summary.groupby(level=0).sum()

,参数组合数,为负的组合
中证500,9,7
原油,9,9
恒生,9,9
标普500,9,8
沪深300,9,7
纳指100,9,7
黄金,9,7


A股的窗口列对应 2 / 4 / 6 年（×0.5 / ×1 / ×1.5）。为负的组合越接近全部，说明信号越不依赖原文挑选的参数。

**判读**：多数参数组合下「信号后 5 日收益低于正常时期」仍然成立（各资产 9 组中有 7–9 组为负），
看不出原文的参数是专门挑出来的。但窗口拉长到 2 倍时效果普遍减弱，A股回看 6 年时大多转为正或接近零。
方向的稳健并不改变第 2 节的结论：负收益主要来自同一两次危机。

---
# 5 · 原文发表之后（2025-05 起）

样本只有约一年半，只作参考。

In [10]:
rows = {}
for a in ASSETS:
    fwd = forward_returns(PRICE[a]).loc[OUT_SAMPLE]
    sd = align(SIG[a], fwd.index)
    r = PRICE[a].pct_change().dropna().loc[OUT_SAMPLE]
    tr, flat = timing_returns(SIG[a], r)
    rows[a] = {"信号天数": int(sd.sum()), "独立事件": count_episodes(sd),
               "信号后5日均值": fwd[sd][5].mean() if sd.any() else np.nan,
               "正常时期5日均值": fwd[~sd][5].mean(),
               "择时累计收益": (1 + tr).prod() - 1, "持有累计收益": (1 + r).prod() - 1}
pd.DataFrame(rows).T.style.format({"信号天数": "{:.0f}", "独立事件": "{:.0f}", "信号后5日均值": "{:+.2%}",
                                   "正常时期5日均值": "{:+.2%}", "择时累计收益": "{:+.2%}", "持有累计收益": "{:+.2%}"},
                                  na_rep="—")

,信号天数,独立事件,信号后5日均值,正常时期5日均值,择时累计收益,持有累计收益
沪深300,0,0,—,+0.19%,+13.72%,+13.72%
中证500,0,0,—,+0.40%,+25.95%,+25.95%
恒生,0,0,—,+0.14%,+11.50%,+11.50%
标普500,42,1,+0.43%,+0.45%,+31.89%,+37.54%
纳指100,43,1,+0.57%,+0.62%,+40.83%,+52.96%
黄金,51,5,+0.24%,+0.38%,+16.41%,+24.99%
原油,17,2,+7.71%,+0.99%,+31.77%,+99.70%


**判读**：发表后约一年半里，A股、恒生没有出现信号；美股、黄金、原油的信号之后没有出现下跌，
空仓反而错过了上涨，择时收益均低于一直持有。样本很短，只能说没有看到信号有效的证据。

---
# 结论

| 问题 | 结论 |
|---|---|
| 原文结果能否复现 | **大部分可以**。标普500、纳指100、原油接近原文；中证500 更强；沪深300、恒生方向一致、幅度较小；**黄金不能复现** |
| 统计上是否可靠 | **效果几乎都来自单次危机**。除恒生外，去掉贡献最大的一次事件（2015 股灾、2020 年 3 月、2022 加息），信号后收益不再为负 |
| 是否存在前视 | 国内金价同日对齐时有前视，原文黄金的短期负收益很可能来自于此；恒生、原油不受影响 |
| 参数是否事后挑选 | 看不出：多数参数组合下方向不变 |
| 原文发表后 | 一年半内没有看到信号有效的证据，择时均跑输持有 |

详见本目录 README。